# Practical Exam: Supermarket Loyalty

International Essentials is an international supermarket chain.

Shoppers at their supermarkets can sign up for a loyalty program that provides rewards each year to customers based on their spending. The more you spend the bigger the rewards. 

The supermarket would like to be able to predict the likely amount customers in the program will spend, so they can estimate the cost of the rewards. 

This will help them to predict the likely profit at the end of the year.

## Data

The dataset contains records of customers for their last full year of the loyalty program.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
|customer_id | Unique identifier for the customer. </br>Missing values are not possible due to the database structure. |
|spend | Continuous. </br>The total spend of the customer in their last full year. This can be any positive value to two decimal places. </br>Missing values should be replaced with 0. |
|first_month | Continuous. </br>The amount spent by the customer in their first month of the year. This can be any positive value, rounded to two decimal places. </br>Missing values should be replaced with 0. |
| items_in_first_month | Discrete. </br>The number of items purchased in the first month. Any integer value greater than or equal to zero. </br>Missing values should be replaced by 0. |  
| region | Nominal. </br>The geographic region that the customer is based in. One of four values Americas, Asia/Pacific, Europe, Middle East/Africa. </br>Missing values should be replaced with "Unknown". |
| loyalty_years | Oridinal. </br>The number of years the customer has been a part of the loyalty program. One of five ordered categories, '0-1', '1-3', '3-5', '5-10', '10+'. </br>Missing values should be replaced with '0-1'.|
| joining_month | Nominal. </br>The month the customer joined the loyalty program. One of 12 values "Jan", "Feb", "Mar", "Apr", etc. </br>Missing values should be replaced with "Unknown".|
| promotion | Nominal. </br>Did the customer join the loyalty program as part of a promotion? Either 'Yes' or 'No'. </br>Missing values should be replaced with 'No'.|


# Task 1

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "loyalty.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
|customer_id | Unique identifier for the customer. </br>Missing values are not possible due to the database structure. |
|spend | Continuous. </br>The total spend of the customer in their last full year. This can be any positive value to two decimal places. </br>Missing values should be replaced with 0. |
|first_month | Continuous. </br>The amount spent by the customer in their first month of the year. This can be any positive value, rounded to two decimal places. </br>Missing values should be replaced with 0. |
| items_in_first_month | Discrete. </br>The number of items purchased in the first month. Any integer value greater than or equal to zero. </br>Missing values should be replaced by 0. |  
| region | Nominal. </br>The geographic region that the customer is based in. One of four values Americas, Asia/Pacific, Europe, Middle East/Africa. </br>Missing values should be replaced with "Unknown". |
| loyalty_years | Oridinal. </br>The number of years the customer has been a part of the loyalty program. One of five ordered categories, '0-1', '1-3', '3-5', '5-10', '10+'. </br>Missing values should be replaced with '0-1'.|
| joining_month | Nominal. </br>The month the customer joined the loyalty program. One of 12 values "Jan", "Feb", "Mar", "Apr", etc. </br>Missing values should be replaced with "Unknown".|
| promotion | Nominal. </br>Did the customer join the loyalty program as part of a promotion? Either 'Yes' or 'No'. </br>Missing values should be replaced with 'No'.|

In [1]:
#library(tidyverse)
#library(caret)
#library(tidymodels)

In [2]:
#data <- read_csv("loyalty.csv")

In [3]:
import pandas as pd

data = pd.read_csv("loyalty.csv")
import numpy as np

# Copia del DataFrame original
clean_data = data.copy()

# Reemplazo de "." por 0 en 'first_month'
clean_data['first_month'] = clean_data['first_month'].replace('.', 0)

# Conversión a numérico
clean_data['first_month'] = pd.to_numeric(clean_data['first_month'], errors='coerce')

# Conversión de 'region' a categoría con niveles específicos
region_levels = ['Americas', 'Asia/Pacific', 'Europe', 'Middle East/Africa']
clean_data['region'] = pd.Categorical(clean_data['region'], categories=region_levels, ordered=False)

# Conversión de 'loyalty_years' a categoría ordenada
loyalty_levels = ['0-1', '1-3', '3-5', '5-10', '10+']
clean_data['loyalty_years'] = pd.Categorical(clean_data['loyalty_years'], categories=loyalty_levels, ordered=True)

# Reemplazo de NA por "Unknown" en 'joining_month'
clean_data['joining_month'] = clean_data['joining_month'].fillna("Unknown")

# Normalización de valores en 'promotion'
clean_data['promotion'] = clean_data['promotion'].replace({
    "YES": "Yes",
    "NO": "No"
})

# Conversión de 'promotion' a categoría con niveles específicos
clean_data['promotion'] = pd.Categorical(clean_data['promotion'], categories=["No", "Yes"], ordered=False)

# Conversión de 'joining_month' a categoría con niveles específicos
month_levels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Unknown"]
clean_data['joining_month'] = pd.Categorical(clean_data['joining_month'], categories=month_levels, ordered=False)

# Resumen del DataFrame
print(clean_data.describe(include='all'))

# Niveles de 'joining_month'
print("\nNiveles de 'joining_month':")
print(clean_data['joining_month'].cat.categories.tolist())


        customer_id        spend  ...  joining_month  promotion
count   1246.000000  1246.000000  ...           1246       1246
unique          NaN          NaN  ...             13          2
top             NaN          NaN  ...            Jan         No
freq            NaN          NaN  ...            146        635
mean     623.500000   122.637119  ...            NaN        NaN
std      359.833526     9.975102  ...            NaN        NaN
min        1.000000   104.290000  ...            NaN        NaN
25%      312.250000   112.210000  ...            NaN        NaN
50%      623.500000   123.840000  ...            NaN        NaN
75%      934.750000   131.092500  ...            NaN        NaN
max     1246.000000   142.290000  ...            NaN        NaN

[11 rows x 8 columns]

Niveles de 'joining_month':
['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Unknown']


# Task 2 

The team at International Essentials have told you that they have always believed that the number of years in the loyalty scheme is the biggest driver of spend. 

Producing a table showing the difference in the average spend by number of years in the loyalty programme along with the variance to investigate this question for the team.

 - You should start with the data in the file 'loyalty.csv'.

 - Your output should be a data frame named `spend_by_years`. 

 - It should include the three columns `loyalty_years`, `avg_spend`, `var_spend`. 

 - Your answers should be rounded to 2 decimal places.   

In [4]:
# Use this cell to write your code for Task 2

import pandas as pd
import numpy as np

# Agrupar por 'loyalty_years' y calcular media y varianza redondeadas
spend_by_years = (
    data
    .groupby('loyalty_years', observed=True)
    .agg(
        avg_spend=('spend', lambda x: round(x.mean(), 2)),
        var_spend=('spend', lambda x: round(x.var(ddof=1), 2))  # ddof=1 para varianza muestral
    )
    .reset_index()
    .loc[:, ['loyalty_years', 'avg_spend', 'var_spend']]
)

print(spend_by_years)


  loyalty_years  avg_spend  var_spend
0           0-1     110.56       9.30
1           1-3     129.31       9.65
2           10+     117.41      16.72
3           3-5     124.55      11.09
4          5-10     135.15      14.10


# Task 3

Fit a baseline model to predict the spend over the year for each customer.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “test.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `customer_id` and `spend`. The `spend` column must be your predicted values.

In [5]:
# Use this cell to write your code for Task 3

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Cargar datos de entrenamiento
train = pd.read_csv("train.csv")

# Convertir 'loyalty_years' en una categoría ordenada
loyalty_order = ['0-1', '1-3', '3-5', '5-10', '10+']
train['loyalty_years'] = pd.Categorical(train['loyalty_years'], categories=loyalty_order, ordered=True)

# Ajustar el modelo lineal
model = smf.ols('spend ~ loyalty_years * first_month + items_in_first_month + joining_month', data=train).fit()

# Mostrar resumen del modelo
print(model.summary())

# Cargar datos de prueba
test = pd.read_csv("test.csv")

# Crear base_result con columna customer_id y valores iniciales de spend
base_result = test[['customer_id']].copy()
base_result['spend'] = np.arange(1, len(test) + 1)

# Predecir valores de spend usando el modelo
base_result['spend'] = model.predict(test)


                            OLS Regression Results                            
Dep. Variable:                  spend   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 3.632e+05
Date:                Wed, 29 Oct 2025   Prob (F-statistic):               0.00
Time:                        00:51:45   Log-Likelihood:                -223.86
No. Observations:                 996   AIC:                             491.7
Df Residuals:                     974   BIC:                             599.6
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

# Task 4

Fit a comparison model to predict the spend over the year for each customer.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “test.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `customer_id` and `spend`. The `spend` column must be your predicted values.

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Leer datos
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Definir variables predictoras y objetivo
categorical_cols = ['loyalty_years', 'first_month', 'joining_month', 'region']
numeric_cols = ['items_in_first_month']
target_col = 'spend'

# Separar X e y
X_train = train[categorical_cols + numeric_cols]
y_train = train[target_col]
X_test = test[categorical_cols + numeric_cols]

# Crear pipeline con codificación + modelo
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'  # deja pasar las columnas numéricas
)

model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Ajustar modelo
model.fit(X_train, y_train)

# Predecir sobre test
predicted_spend = model.predict(X_test)

# Construir resultado final
compare_result = test[['customer_id']].copy()
compare_result['spend'] = predicted_spend